Imports

In [1]:
import pandas as pd

Reading train data

In [3]:
train = pd.read_csv("../data/train_data.zip")

Sort into target and input depending on time-of-closest approach (TCA).
The data from more than two days ago are supposed to predict those from less than one day ago.

In [ ]:
X_events = []
y = []
event_ids = []

for event_id, group in train.groupby("event_id"): # Slight help by ChatGPT

    group = group.sort_values("time_to_tca", ascending=False)

    inputs = group[group["time_to_tca"] >= 2]
    targets = group[group["time_to_tca"] < 1]

    if inputs.empty or targets.empty:
        continue

    X_events.append(inputs.drop(columns=["event_id", "risk"])) # Add the event, event_id is stored elsewhere, risk is leakage

    target_risk = targets.iloc[-1]["risk"]
    y.append(int(target_risk >= -6)) # risk is 10-log

    event_ids.append(event_id)

In [18]:
print(f"length of y: {len(y)}")
print(f"length of X_events: {len(X_events)}")
print(f"instances of excedeing threshold: {sum(y)}")

length of y: 8293
length of X_events: 8293
instances of excedeing threshold: 66


So, each entry in X_events looks like this and shows all CDMs for that event in chronological order:

In [28]:
X_events[0]

,time_to_tca,mission_id,max_risk_estimate,max_risk_scaling,miss_distance,relative_speed,relative_position_r,relative_position_t,relative_position_n,relative_velocity_r,...,t_sigma_rdot,c_sigma_rdot,t_sigma_tdot,c_sigma_tdot,t_sigma_ndot,c_sigma_ndot,F10,F3M,SSN,AP
9,6.983474,2,-6.601713,13.293159,22902.0,14348.0,-1157.6,-6306.2,21986.3,15.8,...,0.153332,39.695541,0.009370,0.269965,0.003886,0.339406,73.0,77.0,27.0,4.0
10,6.691611,2,-6.603452,13.374242,22966.0,14348.0,-1161.1,-6330.2,22046.3,15.8,...,0.252043,39.617319,0.009373,0.269926,0.003939,0.339393,73.0,77.0,27.0,4.0
11,6.269979,2,-6.217958,426.808532,18785.0,14347.0,-698.8,-5176.4,18044.8,14.4,...,0.126831,2.616749,0.010256,0.050424,0.003963,0.193127,71.0,77.0,23.0,8.0
12,6.042352,2,-6.271078,181.496778,18842.0,14347.0,-700.0,-5192.1,18099.4,14.4,...,0.209633,6.985796,0.009402,0.051577,0.003920,0.193324,71.0,77.0,23.0,8.0
13,5.711716,2,-6.277448,187.525360,19015.0,14347.0,-709.9,-5242.1,18264.8,14.5,...,0.110260,6.878515,0.008628,0.051522,0.004023,0.193324,71.0,77.0,23.0,8.0
14,5.377642,2,-6.278272,190.090568,19137.0,14347.0,-710.3,-5273.6,18382.8,14.5,...,0.184069,6.794965,0.008644,0.051514,0.003754,0.193324,70.0,77.0,11.0,5.0
15,5.028915,2,-6.283246,188.270059,18918.0,14347.0,-714.8,-5213.9,18172.0,14.5,...,0.164835,6.960357,0.007950,0.051552,0.003657,0.193325,70.0,77.0,11.0,5.0
16,4.724355,2,-6.283579,192.100563,19152.0,14347.0,-717.5,-5277.4,18397.2,14.5,...,0.150893,6.826289,0.008083,0.051500,0.003711,0.193325,70.0,77.0,11.0,5.0
17,4.334354,2,-6.524619,1984.965171,18635.0,14347.0,-704.3,-5134.8,17900.0,14.4,...,0.079486,1.110077,0.006910,0.052852,0.003578,0.194233,69.0,77.0,0.0,5.0
18,4.055292,2,-6.270511,404.887321,18789.0,14347.0,-700.7,-5177.3,18048.3,14.4,...,0.097613,3.034205,0.006289,0.053201,0.003467,0.194276,69.0,77.0,0.0,5.0


In [97]:
X_events[0].nunique()

time_to_tca          14
mission_id            1
max_risk_estimate    14
max_risk_scaling     14
miss_distance        14
                     ..
c_sigma_ndot         14
F10                   5
F3M                   1
SSN                   4
AP                    4
Length: 101, dtype: int64

In [98]:
count = sum( 
    any(event.select_dtypes(exclude="number").nunique() > 1) for event in X_events
)

print(count) # Only two, so object_type will count as constant as the amount is negligeble

2


In [56]:
for i, event in enumerate(X_events):
    cat = event.select_dtypes(exclude="number")
    changed = cat.columns[cat.nunique() > 1]

    if len(changed) > 0:
        print(f"\nEvent index: {i}")
        for col in changed:
            print(f"{col}: {cat[col].unique()}")


Event index: 4451
c_object_type: ['TBA' 'PAYLOAD']

Event index: 7582
c_object_type: ['TBA' 'PAYLOAD']


In [107]:
def aggregate(X_events):
    X_agg = []

    all_num = pd.concat([
        event.select_dtypes(include="number")
        for event in X_events
    ])

    constant = []
    varying = []

    for col in all_num.columns: # check which are constant and not
        changes = any(
            event[col].nunique() > 1
            for event in X_events
        )
        if changes:
            varying.append(col)
        else:
            constant.append(col)

    for event in X_events: # aggregate events
        num = event.select_dtypes(include="number")
        cat = event.select_dtypes(exclude="number")

        summary = pd.concat([
            num[constant].iloc[-1],
            num[varying].mean().add_suffix("_mean"),
            num[varying].std().fillna(0).add_suffix("_std"),
            num[varying].iloc[0].add_suffix("_first"),
            num[varying].iloc[-1].add_suffix("_last"),
            cat.iloc[-1] # if c_object changed changed from TBA to other, keep the latter 
        ])

        summary["n_cdms"] = len(event)
        X_agg.append(summary)

    X_agg = pd.DataFrame(X_agg)

    X_agg = pd.get_dummies(
        X_agg,
        columns=["mission_id"],
        dtype=int
    )

    return X_agg

In [108]:
X_agg = aggregate(X_events)

In [109]:
X_agg

,t_span,c_span,time_to_tca_mean,max_risk_estimate_mean,max_risk_scaling_mean,miss_distance_mean,relative_speed_mean,relative_position_r_mean,relative_position_t_mean,relative_position_n_mean,...,mission_id_13.0,mission_id_14.0,mission_id_15.0,mission_id_16.0,mission_id_18.0,mission_id_19.0,mission_id_20.0,mission_id_22.0,mission_id_23.0,mission_id_24.0
0,12.00,2.0,4.821300,-6.334024,384.514822,19442.357143,14347.142857,-767.450000,-5358.150000,18673.964286,...,0,0,0,0,0,0,0,0,0,0
1,1.50,2.0,4.570982,-7.197875,74.126013,26478.466667,13574.000000,29.386667,11155.546667,24014.306667,...,0,0,0,0,0,1,0,0,0,0
2,1.50,2.0,4.021045,-7.089900,148.381372,32825.666667,12093.000000,174.900000,-19276.500000,26569.833333,...,0,0,0,0,0,1,0,0,0,0
3,1.00,2.0,4.423633,-4.762943,4.477724,377.714286,2001.000000,-10.242857,-374.414286,-50.614286,...,0,0,0,0,0,0,0,0,0,0
4,21.00,2.0,4.372502,-4.802989,123.129051,10998.800000,10027.000000,139.213333,-8243.213333,-7281.180000,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8288,4.12,2.0,5.750178,-5.030981,0.544229,1026.142857,14781.000000,-221.885714,120.985714,-994.914286,...,0,0,0,0,0,0,0,0,0,0
8289,9.10,2.0,4.436995,-6.473733,7.707430,21896.400000,13295.466667,373.813333,10682.560000,19105.180000,...,0,0,0,0,0,0,0,0,0,0
8290,9.10,2.0,4.502910,-5.649263,115.734392,3636.625000,10330.000000,-166.718750,2668.887500,-2464.312500,...,0,0,0,0,0,0,0,0,0,0
8291,20.00,2.0,6.512812,-6.255655,3680.984800,3674.000000,14952.000000,-1710.750000,-359.900000,3231.350000,...,0,0,0,0,0,0,0,0,0,0


In [110]:
print(X_agg.isna().sum().sort_values(ascending=False).head(20))


c_rcs_estimate_first    2511
c_rcs_estimate_last     2495
c_rcs_estimate_mean     2490
c_crdot_r_first          912
c_cndot_tdot_first       912
c_cndot_rdot_first       912
c_crdot_t_first          912
c_sigma_rdot_first       912
c_sigma_ndot_first       912
c_sigma_tdot_first       912
c_cndot_n_first          912
c_crdot_n_first          912
c_ctdot_t_first          912
c_ctdot_r_first          912
c_ctdot_rdot_first       912
c_cndot_r_first          912
c_cndot_t_first          912
c_ctdot_n_first          912
t_sigma_tdot_first       906
t_sigma_ndot_first       906
dtype: int64


Some missing features, but I mostly solved it:

In [111]:
missing = X_agg.isna().mean().sort_values(ascending=False)

print(missing.head(20))
print("\nColumns with >25% missing:", (missing > 0.25).sum())
print("\nColumns with >50% missing:", (missing > 0.5).sum())
print("\nColumns with >90% missing:", (missing > 0.9).sum())
print("\nColumns with >99% missing:", (missing > 0.99).sum())
print("Columns with no missing:", (missing == 0).sum())

c_rcs_estimate_first    0.302785
c_rcs_estimate_last     0.300856
c_rcs_estimate_mean     0.300253
c_crdot_r_first         0.109972
c_cndot_tdot_first      0.109972
c_cndot_rdot_first      0.109972
c_crdot_t_first         0.109972
c_sigma_rdot_first      0.109972
c_sigma_ndot_first      0.109972
c_sigma_tdot_first      0.109972
c_cndot_n_first         0.109972
c_crdot_n_first         0.109972
c_ctdot_t_first         0.109972
c_ctdot_r_first         0.109972
c_ctdot_rdot_first      0.109972
c_cndot_r_first         0.109972
c_cndot_t_first         0.109972
c_ctdot_n_first         0.109972
t_sigma_tdot_first      0.109249
t_sigma_ndot_first      0.109249
dtype: float64

Columns with >25% missing: 3

Columns with >50% missing: 0

Columns with >90% missing: 0

Columns with >99% missing: 0
Columns with no missing: 289


In [112]:
print(X_agg.iloc[0].to_string())


t_span                                               12.0
c_span                                                2.0
time_to_tca_mean                                   4.8213
max_risk_estimate_mean                          -6.334024
max_risk_scaling_mean                          384.514822
miss_distance_mean                           19442.357143
relative_speed_mean                          14347.142857
relative_position_r_mean                          -767.45
relative_position_t_mean                         -5358.15
relative_position_n_mean                     18673.964286
relative_velocity_r_mean                        14.628571
relative_velocity_t_mean                    -13791.485714
relative_velocity_n_mean                     -3957.185714
t_time_lastob_start_mean                              1.0
t_time_lastob_end_mean                                0.0
t_recommended_od_span_mean                       3.701429
t_actual_od_span_mean                            3.701429
t_obs_availabl